In [ ]:
import pandas as pd
import numpy as np
import openpyxl
from datetime import datetime
import time
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas.tseries.offsets import QuarterEnd
from statsmodels.tsa.arima.model import ARIMA
from pmdarima import auto_arima

In [ ]:
def parse_date_from_filename(filename):
    match = re.search(r'as of (\w{3} \d{2} \d{4})', filename)
    if match:
        date_str = match.group(1)
        try:
            return datetime.strptime(date_str, '%b %d %Y')
        except ValueError:
            truncated_date = date_str[:9]
            year = "20" + date_str[-2:]
            return datetime.strptime(f"{truncated_date}{year}", '%b %d %Y')
    else:
        raise ValueError(f"Unable to parse date from filename: {filename}")

def extract_data_to_dataframe(folder_path):
    data = []

    # Iterate through all files in the folder
    for filename in os.listdir(folder_path):
        if filename.endswith('.xlsx'):
            file_path = os.path.join(folder_path, filename)
            
            try:
                # Extract date from filename
                date = parse_date_from_filename(filename)
                
                # Read the Excel file
                workbook = openpyxl.load_workbook(file_path, read_only=True, data_only=True)
                sheet = workbook.active
                
                # Assuming the first row is header, start from second row
                rows = list(sheet.rows)
                headers = [cell.value for cell in rows[0]]
                ticker_index = headers.index('Ticker')
                price_index = headers.index('Price')
                
                for row in rows[1:]:
                    ticker = row[ticker_index].value
                    price = row[price_index].value
                    
                    if ticker and price:
                        # Append to data list
                        data.append({
                            'ticker': ticker,
                            'price': price,
                            'year': date.year,
                            'month': date.month,
                            'quarter': (date.month - 1) // 3 + 1
                        })
                
                workbook.close()
            
            except Exception as e:
                print(f"Error reading {filename}: {str(e)}")
    
    # Create DataFrame
    final_df = pd.DataFrame(data)
    
    return final_df

# Usage
folder_path = "tickers"
df = extract_data_to_dataframe(folder_path)

# Display the first few rows and basic info about the DataFrame
print(df.head())
print(df.info())

# Optional: Save to CSV
# df.to_csv('spx_data.csv', index=False)

In [ ]:
df

In [ ]:
def extract_common_tickers(folder_path, num_tickers=100):
    ticker_counts = {}
    file_count = 0

    # Iterate through all files in the folder
    for filename in os.listdir(folder_path):
        if filename.endswith('.xlsx'):
            file_path = os.path.join(folder_path, filename)
            
            try:
                # Read the Excel file
                workbook = openpyxl.load_workbook(file_path, data_only=True)
                sheet = workbook.active
                
                # Assume the ticker column is the first column. Adjust if it's different.
                tickers = [cell.value for cell in sheet['A'] if cell.value and cell.value != 'Ticker']
                
                for ticker in tickers:
                    if ticker in ticker_counts:
                        ticker_counts[ticker] += 1
                    else:
                        ticker_counts[ticker] = 1
                file_count += 1
                
                workbook.close()
            
            except Exception as e:
                print(f"Error reading {filename}: {str(e)}")
    
    # Find tickers present in all files
    common_tickers = [ticker for ticker, count in ticker_counts.items() if count == file_count]
    
    # If we have more than 100 common tickers, select the top 100 by frequency
    if len(common_tickers) > num_tickers:
        common_tickers.sort(key=lambda x: ticker_counts[x], reverse=True)
        common_tickers = common_tickers[:num_tickers]
    
    return common_tickers

# Usage
common_tickers = extract_common_tickers("tickers")
print(f"Number of common tickers found: {len(common_tickers)}")
print("Common tickers:", common_tickers)

In [ ]:
df.info()

In [ ]:
def filter_tickers(df):
    # Define the regex pattern to match tickers starting with 'AAPL'
    pattern = r'^AAPL.*'
    
    # Use the regex pattern to filter the DataFrame
    filtered_df = df[df['ticker'].apply(lambda x: bool(re.match(pattern, x)))]
    
    return filtered_df

# Example usage
filtered_df = filter_tickers(df)
filtered_df

In [ ]:
import pandas as pd

def get_date_range(df, tickers):
    # Dictionary to store the start and end dates for each ticker
    date_ranges = {}
    
    for ticker in tickers:
        # Filter the DataFrame for the specific ticker
        ticker_df = df[df['ticker'] == ticker]
        
        if not ticker_df.empty:
            # Get the minimum and maximum dates for the ticker
            start_date = ticker_df['date'].min()
            end_date = ticker_df['date'].max()
            
            # Store the result in the dictionary
            date_ranges[ticker] = (start_date, end_date)
    
    return date_ranges

# Example usage:
tickers = ['AAPL UQ Equity', 'AAPL UW Equity']
result = get_date_range(df, tickers)
print(result)

In [ ]:
def simplify_tickers(df):
    # Extract the base ticker (without the equity suffix) by splitting at the first space
    df['base_ticker'] = df['ticker'].apply(lambda x: x.split(' ')[0])
    
    # Create a new DataFrame to store the simplified data
    simplified_df = pd.DataFrame()
    
    # Group by base_ticker to process each base ticker individually
    grouped = df.groupby('base_ticker')
    
    for base_ticker, group in grouped:
        # Keep all rows for each base ticker
        group['ticker'] = base_ticker  # Rename the ticker to just the base ticker
        
        # Append the modified group to the simplified DataFrame
        simplified_df = pd.concat([simplified_df, group], ignore_index=True)
    
    # Drop the 'base_ticker' column as it's no longer needed
    simplified_df = simplified_df.drop(columns=['base_ticker'])
    
    return simplified_df

# Example usage:
simplified_df = simplify_tickers(df)

In [ ]:
simplified_df

In [ ]:
unique_tickers = simplified_df['ticker'].unique()

In [ ]:
unique_tickers


In [ ]:
def print_data_for_unique_dates_per_ticker(simplified_df):
    # Get unique tickers from the DataFrame
    accumulated_df = pd.DataFrame()
    unique_tickers = simplified_df['ticker'].unique()
    
    for ticker in unique_tickers:
        # Filter the DataFrame for the current ticker
        ticker_df = simplified_df[simplified_df['ticker'] == ticker]
        
        # Get unique dates for the current ticker
        unique_dates = ticker_df['date'].unique()
        
        # Filter the ticker DataFrame to include only rows with these unique dates
        filtered_ticker_df = ticker_df[ticker_df['date'].isin(unique_dates)]
        if not filtered_ticker_df.empty:
            # Append the filtered data to the accumulated DataFrame
            accumulated_df = pd.concat([accumulated_df, filtered_ticker_df], ignore_index=True)
    return accumulated_df
       
abc=print_data_for_unique_dates_per_ticker(simplified_df)

In [ ]:
abc

In [ ]:
aapl_rows = abc[abc['ticker'] == 'AAPL']
aapl_rows

In [ ]:
import pandas as pd

def filter_aapl_unique_dates(abc):
    # Step 1: Filter rows where the ticker is 'AAPL'
    aapl_df = abc[abc['ticker'] == 'AAPL']
    
    # Step 2: Identify unique dates
    unique_dates = aapl_df['date'].unique()
    
    # Step 3: Filter rows to keep only those with unique dates
    # Drop duplicates based on the 'date' column
    aapl_unique_dates_df = aapl_df.drop_duplicates(subset='date')
    
    return aapl_unique_dates_df

# Example usage:
filtered_aapl_df = filter_aapl_unique_dates(abc)
filtered_aapl_df

In [ ]:
def filter_unique_dates_per_ticker(abc):
    # Initialize an empty DataFrame to accumulate the results
    unique_dates_df = pd.DataFrame()
    
    # Get unique tickers from the DataFrame
    unique_tickers = abc['ticker'].unique()
    
    for ticker in unique_tickers:
        # Filter rows where the ticker matches the current ticker
        ticker_df = abc[abc['ticker'] == ticker]
        
        # Drop duplicates based on the 'date' column
        ticker_unique_dates_df = ticker_df.drop_duplicates(subset='date')
        
        # Append the result to the accumulated DataFrame
        unique_dates_df = pd.concat([unique_dates_df, ticker_unique_dates_df], ignore_index=True)
    
    return unique_dates_df

# Example usage:
unique_dates_per_ticker_df = filter_unique_dates_per_ticker(abc)
unique_dates_per_ticker_df

In [ ]:
## filtered - clean dataframe
unique_dates_per_ticker_df

In [ ]:
import pandas as pd

def split_dataset(df, split_year=2023):
    # Ensure the date column is in datetime format
    df['date'] = pd.to_datetime(df['date'])
    
    # Sort the dataframe by date
    df = df.sort_values('date')
    
    # Split the dataframe
    df_2022 = df[df['date'].dt.year < split_year]
    df_2024 = df[df['date'].dt.year >= split_year]
    
    print(f"Training set (2000 to {split_year-1}):")
    print(f"Start date: {df_2022['date'].min()}")
    print(f"End date: {df_2022['date'].max()}")
    print(f"Number of records: {len(df_2022)}")
    
    print(f"\nTest set ({split_year} to last date):")
    print(f"Start date: {df_2024['date'].min()}")
    print(f"End date: {df_2024['date'].max()}")
    print(f"Number of records: {len(df_2024)}")
    
    return df_2022, df_2024

df_2022, df_2024 = split_dataset(unique_dates_per_ticker_df)

In [ ]:
aapl_rows = unique_dates_per_ticker_df[unique_dates_per_ticker_df['ticker'] == 'AAPL']
aapl_rows

In [ ]:
import pandas as pd

def print_data_for_tickers_and_unique_dates(df):
    # Get unique tickers
    unique_tickers = df['ticker'].unique()
    
    for ticker in unique_tickers:
        # Filter the DataFrame for the current ticker
        ticker_df = df[df['ticker'] == ticker]
        
        # Get unique dates for the current ticker
        unique_dates = ticker_df['date'].unique()
        
        # Filter the ticker DataFrame to include only rows with unique dates
        filtered_ticker_df = ticker_df[ticker_df['date'].isin(unique_dates)]
        
        print(f"Data for Ticker: {ticker}")
        print("="*50)
        
        if not filtered_ticker_df.empty:
            # Print the data for the current ticker
            print(filtered_ticker_df)
        else:
            print("No data available for these dates.")
        
        print("-"*50)

# Example usage:
print_data_for_tickers_and_unique_dates(df)


In [ ]:
unique_dates = simplified_df['date'].unique()
print(unique_dates)

In [ ]:
simplified_df['ticker'].unique()

In [ ]:
def print_data_for_unique_dates(df):
    # Store unique dates
    unique_dates = df['date'].unique()
    
    # Get unique tickers
    unique_tickers = df['ticker'].unique()
    
    for ticker in unique_tickers:
        # Filter the DataFrame for the current ticker
        ticker_df = df[df['ticker'] == ticker]
        
        # Further filter ticker_df to include only rows with the unique dates
        filtered_ticker_df = ticker_df[ticker_df['date'].isin(unique_dates)]
        
        print(f"Data for Ticker: {ticker}")
        print("="*50)
        
        if not filtered_ticker_df.empty:
            # Print the data for the current ticker
            print(filtered_ticker_df)
        else:
            print("No data available for these unique dates.")
        
        print("-"*50)

# Example usage:
print_data_for_unique_dates(simplified_df)

In [ ]:
unique_date_rows = simplified_df.groupby('date').filter(lambda x: len(x) == 1)

# Print the rows with unique dates
unique_date_rows

## Analysing and Forecasting ARIMA Model with 2000-2022 and 2023-2024 data

In [ ]:
def calculate_percentage_change(start_value, end_value):
    """Calculate percentage change between two values."""
    return ((end_value - start_value) / start_value) * 100

def analyze_and_forecast_stocks(data, df_2024, num_stocks, forecast_quarters):
    # Ensure date is in datetime format for both dataframes
    data['date'] = pd.to_datetime(data['date'])
    df_2024['date'] = pd.to_datetime(df_2024['date'])
    
    # Get the latest market cap for each stock and sort
    latest_market_cap = data.groupby('ticker')['CUR_MKT_CAP'].last().sort_values(ascending=False)
    top_stocks = latest_market_cap.head(num_stocks).index.tolist()

    def format_date_with_quarter(date):
        year = date.year
        quarter = (date.month - 1) // 3 + 1
        return f"{year}-Q{quarter}"

    # Dictionary to store differences between ARIMA and Actual percentage changes
    difference_dict = {}

    for ticker in top_stocks:
        print(f"\nAnalyzing {ticker}")
        
        # Filter data for this ticker
        stock_data = data[data['ticker'] == ticker].sort_values('date')
        stock_data_2024 = df_2024[df_2024['ticker'] == ticker].sort_values('date')
        
        # Resample to quarterly data
        quarterly_data = stock_data.set_index('date')['PX_LAST'].resample('Q').last().dropna()
        quarterly_data_2024 = stock_data_2024.set_index('date')['PX_LAST'].resample('Q').last().dropna()
        
        if len(quarterly_data) < 8:
            print(f"Not enough data for {ticker}. Skipping.")
            continue

        # Calculate Simple Moving Average (SMA)
        ma_window = 4  # 1-year moving average
        sma = quarterly_data.rolling(window=ma_window).mean()

        # Calculate Exponential Moving Average (EMA)
        ema = quarterly_data.ewm(span=ma_window, adjust=False).mean()

        # ARIMA Forecasting
        try:
            # Automatically select ARIMA parameters
            auto_model = auto_arima(quarterly_data, seasonal=True, m=4, suppress_warnings=True, error_action="ignore")
            
            # Fit the ARIMA model
            model = ARIMA(quarterly_data, order=auto_model.order, seasonal_order=auto_model.seasonal_order)
            results = model.fit()

            # Generate forecast
            forecast = results.forecast(steps=forecast_quarters)
            forecast_dates = pd.date_range(start=quarterly_data.index[-1] + QuarterEnd(), periods=forecast_quarters, freq='Q')
            forecast.index = forecast_dates

            # Plot results
            plt.figure(figsize=(12, 6))
            plt.plot(quarterly_data.index, quarterly_data.values, label='Historical Data (2000-2022)')
            plt.plot(quarterly_data_2024.index, quarterly_data_2024.values, label='Actual Data (2023-2024)', color='purple')
            plt.plot(sma.index, sma.values, label='Simple Moving Average', color='orange')
            plt.plot(ema.index, ema.values, label='Exponential Moving Average', color='green')
            plt.plot(forecast.index, forecast.values, label='ARIMA Forecast', color='red')
            plt.title(f'{ticker} Stock Price, Moving Averages, and ARIMA Forecast')
            plt.xlabel('Date')
            plt.ylabel('Price')
            plt.legend()
            plt.grid(True)
            plt.show()

            # Calculate percentage changes
            arima_change = calculate_percentage_change(forecast.iloc[0], forecast.iloc[-1])
            actual_change = calculate_percentage_change(quarterly_data_2024.iloc[0], quarterly_data_2024.iloc[-1])

            # Calculate the absolute difference between ARIMA and Actual changes
            difference = abs(arima_change - actual_change)
            difference_dict[ticker] = difference

            # Print forecast values
            print(f"\nARIMA Forecast for {ticker}:")
            for date, price in forecast.items():
                formatted_date = format_date_with_quarter(date)
                print(f"{formatted_date}: ${price:.2f}")

            # Print some statistics
            print(f"Data range: ${quarterly_data.min():.2f} to ${quarterly_data.max():.2f}")
            print(f"Number of quarters: {len(quarterly_data)}")
            print(f"Current price (2022): ${quarterly_data.iloc[-1]:.2f}")
            print(f"Latest price (2024): ${quarterly_data_2024.iloc[-1]:.2f}")
            print(f"Current SMA: ${sma.iloc[-1]:.2f}")
            print(f"Current EMA: ${ema.iloc[-1]:.2f}")
            print(f"ARIMA Model: {auto_model.order}, Seasonal: {auto_model.seasonal_order}")
            print(f"ARIMA Forecast % Change: {arima_change:.2f}%")
            print(f"Actual 2023-2024 % Change: {actual_change:.2f}%")
            print(f"Difference in % Change: {difference:.2f}%")

        except Exception as e:
            print(f"Error in ARIMA forecasting for {ticker}: {str(e)}")

    # After analyzing all stocks, sort the difference dictionary
    sorted_differences = sorted(difference_dict.items(), key=lambda x: x[1])

    # Print the 25 stocks with the least difference
    print("\nTop 25 stocks with least difference between ARIMA and Actual percentage changes:")
    for ticker, diff in sorted_differences[:25]:
        print(f"{ticker}: {diff:.2f}%")

    return difference_dict

# Example usage:
# Assuming df_2022 contains data from 2000 to 2022 and df_2024 contains data for 2023-2024
difference_dict = analyze_and_forecast_stocks(df_2022, df_2024, num_stocks=50, forecast_quarters=4)

In [ ]:
sorted_dict = sorted(difference_dict.items(), key=lambda x: x[1])
sorted_dict

## Momentum Indicator

In [ ]:
def calculate_momentum_indicators(data, sorted_dict, num_stocks=25):
    # Get the top 25 stocks from the sorted dictionary
    top_stocks = list(dict(sorted_dict[:num_stocks]).keys())

    for ticker in top_stocks:
        print(f"\nAnalyzing {ticker}")
        
        # Filter data for this ticker
        stock_data = data[data['ticker'] == ticker].sort_values('date')
        
        # Resample to daily data (assuming we have daily data)
        daily_data = stock_data.set_index('date')['PX_LAST'].resample('D').last().dropna()
        
        if len(daily_data) < 50:
            print(f"Not enough data for {ticker}. Skipping.")
            continue

        # Calculate RSI
        delta = daily_data.diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))

        # Calculate MACD
        exp1 = daily_data.ewm(span=12, adjust=False).mean()
        exp2 = daily_data.ewm(span=26, adjust=False).mean()
        macd = exp1 - exp2
        signal = macd.ewm(span=9, adjust=False).mean()
        histogram = macd - signal

        # Calculate ROC
        roc = daily_data.pct_change(periods=12) * 100

        # Plot results
        fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(12, 20), sharex=True)
        
        # Price plot
        ax1.plot(daily_data.index, daily_data.values)
        ax1.set_title(f'{ticker} Stock Price and Momentum Indicators')
        ax1.set_ylabel('Price')

        # RSI plot
        ax2.plot(rsi.index, rsi.values)
        ax2.set_ylabel('RSI')
        ax2.axhline(y=70, color='r', linestyle='--')
        ax2.axhline(y=30, color='g', linestyle='--')
        ax2.set_ylim(0, 100)

        # MACD plot
        ax3.plot(macd.index, macd.values, label='MACD')
        ax3.plot(signal.index, signal.values, label='Signal Line')
        ax3.bar(histogram.index, histogram.values, label='Histogram')
        ax3.set_ylabel('MACD')
        ax3.legend()

        # ROC plot
        ax4.plot(roc.index, roc.values)
        ax4.set_ylabel('ROC')
        ax4.axhline(y=0, color='r', linestyle='--')

        plt.xlabel('Date')
        plt.tight_layout()
        plt.show()

        # Print some statistics
        print(f"Current price: ${daily_data.iloc[-1]:.2f}")
        print(f"Current RSI: {rsi.iloc[-1]:.2f}")
        print(f"Current MACD: {macd.iloc[-1]:.2f}")
        print(f"Current Signal Line: {signal.iloc[-1]:.2f}")
        print(f"Current ROC: {roc.iloc[-1]:.2f}%")

# Example usage:
# Assuming you have run the previous ARIMA analysis and have the sorted_differences
# and the full dataset (unique_dates_per_ticker_df) available

# Then, run the momentum indicators analysis on the top 25 stocks
calculate_momentum_indicators(df_2022, sorted_dict, num_stocks=25)

## Black-Litterman Portfolio Allocation

In [ ]:
def black_litterman_portfolio(data, sorted_dict, risk_free_rate=0.03, market_risk_premium=0.05, tau=0.05, num_stocks=25):
    # Get the top 25 stocks from the sorted dictionary
    top_stocks = list(dict(sorted_dict[:num_stocks]).keys())
    
    # Filter data for top stocks
    stock_data = data[data['ticker'].isin(top_stocks)]
    
    # Calculate returns
    returns = stock_data.pivot(index='date', columns='ticker', values='PX_LAST').pct_change().dropna()
    
    # Ensure we have data for all stocks
    if len(returns.columns) < num_stocks:
        missing_stocks = set(top_stocks) - set(returns.columns)
        print(f"Missing data for stocks: {missing_stocks}")
        top_stocks = list(returns.columns)
        num_stocks = len(top_stocks)
    
    if num_stocks == 0:
        print("No valid stocks to analyze.")
        return None
    
    # Calculate market capitalizations
    market_caps = stock_data.groupby('ticker')['CUR_MKT_CAP'].last()
    market_caps = market_caps[top_stocks]  # Ensure order matches returns
    
    # Calculate market weights
    market_weights = market_caps / market_caps.sum()
    
    # Calculate covariance matrix
    cov_matrix = returns.cov()
    
    # Calculate implied excess returns
    implied_returns = risk_free_rate + market_risk_premium * (cov_matrix @ market_weights)
    
    # Investor views (example: expecting 20% of stocks to outperform by 5%)
    num_views = max(1, min(int(0.2 * num_stocks), num_stocks - 1))  # Ensure we have at least one view
    P = np.zeros((num_views, num_stocks))
    Q = np.zeros(num_views)
    for i in range(num_views):
        P[i, i] = 1
        P[i, -(i+1)] = -1
        Q[i] = 0.05  # 5% outperformance
    
    # Calculate omega (uncertainty in views)
    omega = np.dot(np.dot(P, tau * cov_matrix), P.T) * np.eye(num_views)
    
    # Compute posterior returns
    pi = implied_returns.values
    
    # Debug information
    print(f"Number of stocks: {num_stocks}")
    print(f"Shape of pi: {pi.shape}")
    print(f"Shape of cov_matrix: {cov_matrix.shape}")
    print(f"Shape of P: {P.shape}")
    print(f"Shape of Q: {Q.shape}")
    print(f"Shape of omega: {omega.shape}")
    
    try:
        term1 = tau * cov_matrix @ P.T
        print(f"Shape of term1: {term1.shape}")
        term2 = P @ tau * cov_matrix @ P.T + omega
        print(f"Shape of term2: {term2.shape}")
        term3 = np.linalg.inv(term2)
        print(f"Shape of term3: {term3.shape}")
        term4 = Q - P @ pi
        print(f"Shape of term4: {term4.shape}")
        returns_bl = pi + term1 @ term3 @ term4
    except np.linalg.LinAlgError as e:
        print(f"Error in matrix inversion: {e}")
        print("Using implied returns as posterior returns.")
        returns_bl = pi
    except ValueError as e:
        print(f"ValueError in calculations: {e}")
        print("Using implied returns as posterior returns.")
        returns_bl = pi
    
    # Compute Black-Litterman covariance
    try:
        cov_bl = cov_matrix + np.linalg.inv(tau * np.linalg.inv(cov_matrix) + P.T @ np.linalg.inv(omega) @ P)
    except np.linalg.LinAlgError:
        print("Error in covariance calculation. Using original covariance matrix.")
        cov_bl = cov_matrix
    
    # Define objective function for portfolio optimization
    def objective(weights):
        portfolio_return = np.sum(returns_bl * weights)
        portfolio_volatility = np.sqrt(np.dot(weights.T, np.dot(cov_bl, weights)))
        return -portfolio_return / portfolio_volatility  # Negative Sharpe ratio
    
    # Define constraints
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds = tuple((0, 1) for _ in range(num_stocks))
    
    # Perform optimization
    result = optimize.minimize(objective, market_weights, method='SLSQP', bounds=bounds, constraints=constraints)
    
    # Extract optimal weights
    optimal_weights = pd.Series(result.x, index=top_stocks)
    
    # Calculate expected return and volatility
    expected_return = np.sum(returns_bl * optimal_weights)
    expected_volatility = np.sqrt(np.dot(optimal_weights.T, np.dot(cov_bl, optimal_weights)))
    sharpe_ratio = expected_return / expected_volatility
    
    print("\nBlack-Litterman Portfolio Allocation:")
    for stock, weight in optimal_weights.items():
        print(f"{stock}: {weight:.4f}")
    # print(f"\nExpected Annual Return: {expected_return*252:.2%}")
    # print(f"Expected Annual Volatility: {expected_volatility*np.sqrt(252):.2%}")
    print(f"Sharpe Ratio: {sharpe_ratio*np.sqrt(252):.2f}")
    
    # Plot allocation graph
    plt.figure(figsize=(12, 6))
    optimal_weights.sort_values(ascending=True).plot(kind='barh')
    plt.title('Black-Litterman Portfolio Allocation')
    plt.xlabel('Weight')
    plt.ylabel('Stock')
    plt.tight_layout()
    plt.show()
    
    # # Plot expected returns graph
    # expected_returns = pd.Series(returns_bl, index=top_stocks)
    # plt.figure(figsize=(12, 6))
    # expected_returns.sort_values(ascending=True).plot(kind='barh')
    # plt.title('Expected Returns of Stocks in Portfolio')
    # plt.xlabel('Expected Return')
    # plt.ylabel('Stock')
    # plt.tight_layout()
    # plt.show()

    expected_returns = pd.Series(returns_bl * 252, index=top_stocks)  # Assuming 252 trading days
    
    # Plot risk-return scatter plot
    plt.figure(figsize=(12, 8))
    volatilities = np.sqrt(np.diag(cov_bl))
    plt.scatter(volatilities, expected_returns, s=optimal_weights*1000, alpha=0.5)
    for i, stock in enumerate(top_stocks):
        plt.annotate(stock, (volatilities[i], expected_returns[i]))
    plt.title('Risk-Return Profile of Stocks in Portfolio')
    plt.xlabel('Volatility')
    plt.ylabel('Expected Return')
    plt.tight_layout()
    plt.show()

    return optimal_weights

# Example usage:
# Then run the Black-Litterman portfolio optimization
optimal_weights = black_litterman_portfolio(df_2024, sorted_dict, num_stocks=25)